# RAG Pipeline: Document Assistant

This notebook builds a practical retrieval-augmented generation pipeline over the PDFs in `../data`. Run the cells from top to bottom. Ollama and Chroma are optional at import time, but are required for local generation and persistent vector search respectively.

In [2]:
# 1. Project Configuration
from pathlib import Path
import json
import re
import hashlib

PROJECT_ROOT = Path.cwd().resolve().parent
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"
VECTOR_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
TOP_K = 4
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
OLLAMA_MODEL = "llama3.2"
PDF_FILES = sorted(DATA_DIR.glob("*.pdf"))

print(f"Project root: {PROJECT_ROOT}")
print(f"PDFs found: {len(PDF_FILES)}")

Project root: E:\Project_RAG\rag-assistant-project
PDFs found: 7


In [3]:
# 3. Text Extraction
# Keep page boundaries so citations can point back to an approximate page.
try:
    from pypdf import PdfReader
except ImportError:
    PdfReader = None

extraction_failures = []
def extract_page_text(page, source, page_number):
    try:
        return page.extract_text() or ""
    except Exception as error:
        extraction_failures.append({
            "source": source,
            "page": page_number,
            "error": f"{type(error).__name__}: {error}",
        })
        return ""

page_documents = []
if PdfReader is not None:
    for pdf_path in PDF_FILES:
        reader = PdfReader(str(pdf_path))
        for page_number, page in enumerate(reader.pages, start=1):
            page_documents.append({
                "source": pdf_path.name,
                "page": page_number,
                "text": extract_page_text(page, pdf_path.name, page_number),
            })
print(f"Extracted {len(page_documents)} pages")
print(f"Pages skipped because of extraction errors: {len(extraction_failures)}")
print(page_documents[0]["text"][:500] if page_documents else "No pages extracted")

Extracted 6782 pages
Pages skipped because of extraction errors: 92



In [4]:
# 4. Text Cleaning
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for item in page_documents:
    item["text"] = clean_text(item["text"])
page_documents = [item for item in page_documents if item["text"]]
print(f"Non-empty pages after cleaning: {len(page_documents)}")

Non-empty pages after cleaning: 6604


In [5]:
# 5. Chunking Strategy
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    step = max(1, chunk_size - overlap)
    return [" ".join(words[start:start + chunk_size]) for start in range(0, len(words), step) if words[start:start + chunk_size]]

chunks = []
for page in page_documents:
    for chunk_number, chunk in enumerate(chunk_text(page["text"]), start=1):
        chunks.append({"text": chunk, "source": page["source"], "page": page["page"], "chunk": chunk_number})
print(f"Created {len(chunks)} chunks")

Created 6651 chunks


In [6]:
# 6. Metadata Creation
for index, item in enumerate(chunks):
    item["id"] = hashlib.sha1(f"{item['source']}:{item['page']}:{item['chunk']}".encode()).hexdigest()[:16]
    item["metadata"] = {"source": item["source"], "page": item["page"], "chunk": item["chunk"]}

print(json.dumps(chunks[0] if chunks else {}, indent=2)[:800])

{
  "text": "Praise for AI Engineering This book offers a comprehensive, well-structured guide to the essential aspects of building generative AI systems. A must-read for any professional looking to scale AI across the enterprise. \u2014Vittorio Cretella, former global CIO, P&G and Mars Chip Huyen gets generative AI. On top of that, she is a remarkable teacher and writer whose work has been instrumental in helping teams bring AI into production. Drawing on her deep expertise, AI Engineering serves as a comprehensive and holistic guide, masterfully detailing everything required to design and deploy generative AI applications in production. \u2014Luke Metz, cocreator of ChatGPT, former research manager at OpenAI Every AI engineer building real-world applications should read this book. It\u20


In [32]:
# 7. Embedding Model
embedding_backend = "sentence-transformers"
try:
    from sentence_transformers import SentenceTransformer
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    def embed_texts(texts):
        return embedding_model.encode(texts, normalize_embeddings=True).tolist()
    print(f"Loaded {EMBEDDING_MODEL}")
except Exception as sentence_error:
    try:
        from sklearn.feature_extraction.text import TfidfVectorizer
        embedding_backend = "tfidf"
        tfidf_vectorizer = TfidfVectorizer(max_features=4096, stop_words="english")
        tfidf_is_fitted = False
        def embed_texts(texts):
            nonlocal_placeholder = None
            global tfidf_is_fitted
            if not tfidf_is_fitted:
                vectors = tfidf_vectorizer.fit_transform(texts)
                tfidf_is_fitted = True
            else:
                vectors = tfidf_vectorizer.transform(texts)
            return vectors.toarray().tolist()
        print(f"SentenceTransformers unavailable ({sentence_error}); using TF-IDF embeddings instead.")
    except Exception as tfidf_error:
        embedding_backend = "hash"
        print(f"SentenceTransformers unavailable ({sentence_error}); TF-IDF unavailable ({tfidf_error}); using hash embeddings.")
        def embed_texts(texts, dimensions=384):
            vectors = []
            for text in texts:
                vector = [0.0] * dimensions
                for token in re.findall(r"[a-z0-9]+", text.lower()):
                    vector[int(hashlib.md5(token.encode()).hexdigest(), 16) % dimensions] += 1.0
                norm = sum(value * value for value in vector) ** 0.5 or 1.0
                vectors.append([value / norm for value in vector])
            return vectors

chunk_embeddings = embed_texts([item["text"] for item in chunks]) if chunks else []
print(f"Embedding backend: {embedding_backend}")
print(f"Embedding vectors: {len(chunk_embeddings)}")

SentenceTransformers unavailable ([WinError 4551] An Application Control policy has blocked this file. Error loading "c:\Users\dt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\lib\torch_global_deps.dll" or one of its dependencies.); using TF-IDF embeddings instead.
Embedding backend: tfidf
Embedding vectors: 6651


In [33]:
# 8. Chroma Vector Database
try:
    import chromadb
    chroma_client = chromadb.PersistentClient(path=str(VECTOR_DIR))
    collection_name = f"rag_documents_{embedding_backend}"
    collection = chroma_client.get_or_create_collection(collection_name)
    chroma_batch_size = 5000
    for start in range(0, len(chunks), chroma_batch_size):
        end = start + chroma_batch_size
        batch = chunks[start:end]
        collection.upsert(
            ids=[item["id"] for item in batch],
            documents=[item["text"] for item in batch],
            metadatas=[item["metadata"] for item in batch],
            embeddings=chunk_embeddings[start:end],
        )
    print(f"Chroma collection '{collection_name}' contains {collection.count()} chunks")
except Exception as error:
    chromadb = None
    collection = None
    print(f"Chroma unavailable ({error}); retrieval will use in-memory cosine search.")

Chroma collection 'rag_documents_tfidf' contains 6651 chunks


In [56]:
# 9. Retrieval Function
MAX_DISTANCE = 0.55 if embedding_backend == "sentence-transformers" else 0.99
STOP_WORDS = {"what", "is", "the", "a", "an", "of", "in", "on", "to", "for", "and", "or", "with", "how", "why", "can", "does", "do", "are", "about", "give", "explain", "using", "these", "books", "answer"}

def retrieval_terms(text):
    return {term for term in re.findall(r"[a-z0-9]+", text.lower()) if term not in STOP_WORDS and len(term) > 2}

def retrieve(query, top_k=TOP_K, diverse_sources=False):
    if not chunks:
        return []
    query_terms = retrieval_terms(query)
    query_tokens = re.findall(r"[A-Za-z0-9]+", query)
    required_terms = {token.lower() for token in query_tokens[1:] if token[:1].isupper()}
    minimum_term_matches = 2 if len(query_terms) >= 3 else 1
    query_embedding = embed_texts([query])[0]
    candidate_k = max(top_k * 20, top_k) if embedding_backend == "tfidf" else top_k
    if diverse_sources:
        candidate_k = max(candidate_k, top_k * len(PDF_FILES))

    if collection is not None and embedding_backend != "tfidf":
        result = collection.query(query_embeddings=[query_embedding], n_results=candidate_k)
        results = [{"text": text, "metadata": metadata, "distance": distance}
                   for text, metadata, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0])]
    else:
        scores = [sum(a * b for a, b in zip(query_embedding, vector)) for vector in chunk_embeddings]
        ranked = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)[:candidate_k]
        results = [{"text": chunks[index]["text"], "metadata": chunks[index]["metadata"], "distance": 1 - scores[index]} for index in ranked]

    results = [
        result for result in results
        if len(result["text"]) >= 200
        and result["distance"] <= MAX_DISTANCE
        and len(query_terms & retrieval_terms(result["text"])) >= minimum_term_matches
        and required_terms <= retrieval_terms(result["text"])
    ]
    results.sort(
        key=lambda result: (
            -sum(result["text"].lower().count(term) for term in query_terms),
            result["distance"],
        )
    )
    if diverse_sources:
        unique_results = []
        seen_sources = set()
        for result in results:
            source = result["metadata"]["source"]
            if source not in seen_sources:
                unique_results.append(result)
                seen_sources.add(source)
            if len(unique_results) == top_k:
                break
        return unique_results
    return results[:top_k]

retrieved = retrieve("What is retrieval augmented generation?")
print([(item["metadata"], round(item["distance"], 3)) for item in retrieved])
print("Sources returned:", sorted({item["metadata"]["source"] for item in retrieved}))

[({'source': 'AI Engineering by Chip Huyen.pdf', 'page': 978, 'chunk': 1}, 0.594), ({'source': 'AI Engineering by Chip Huyen.pdf', 'page': 980, 'chunk': 1}, 0.517), ({'source': 'Hands-on Large Language Models.pdf', 'page': 271, 'chunk': 1}, 0.558), ({'source': 'Hands-on Large Language Models.pdf', 'page': 10, 'chunk': 1}, 0.67)]
Sources returned: ['AI Engineering by Chip Huyen.pdf', 'Hands-on Large Language Models.pdf']


In [13]:
# 10. Ollama LLM
try:
    import ollama
    ollama_available = True
    print(f"Ollama client ready for model: {OLLAMA_MODEL}")
except ImportError:
    ollama = None
    ollama_available = False
    print("Install ollama and start the local service to generate answers.")

def generate_with_ollama(prompt):
    if ollama is None:
        return "Ollama is unavailable. Review the retrieved context below."
    try:
        response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
        return response["message"]["content"]
    except Exception as error:
        return f"Ollama request failed: {error}"

Ollama client ready for model: llama3.2


In [61]:
# 11. RAG Pipeline
def answer_question(question, top_k=TOP_K):
    sources = retrieve(question, top_k=top_k)
    if not sources:
        return {
            "question": question,
            "answer": "The answer is not supported by the provided documents.",
            "sources": [],
        }

    context = "\n\n".join(
        f"[{item['metadata']['source']}, page {item['metadata']['page']}] {item['text']}"
        for item in sources
    )
    prompt = f"""You are a closed-book document question-answering assistant.
Use ONLY the provided context. Do not use outside knowledge or guess.
If the context does not directly support the answer, respond exactly:
The answer is not supported by the provided documents.

For a supported answer, write 3 to 5 complete sentences:
1. Start with a direct definition or answer.
2. Explain the main idea using the context.
3. Explain how it works or why it matters when the context supports this.
4. End every factual sentence with a citation in this format: [filename, page N].
Do not return only a title, abbreviation, or fragment. Never omit citations.

Context:
{context}

Question: {question}
Answer:"""
    answer = generate_with_ollama(prompt)
    unsupported_markers = (
        "not supported by the provided documents",
        "does not directly support",
        "not explicitly mentioned",
        "cannot be determined",
    )
    if any(marker in answer.lower() for marker in unsupported_markers):
        sources = []
    return {"question": question, "answer": answer, "sources": sources}

sample_result = answer_question("What is the purpose of a machine learning system?")
print(sample_result["answer"])
print([item["metadata"] for item in sample_result["sources"]])

The purpose of a machine learning system is to enable systems to automatically improve performance on tasks through experience and data rather than explicit programming, thus bridging the gap between theoretical machine learning advances and practical deployment in production systems.

This is evident from the context provided in Chapter 12, which defines machine learning as a subset of artificial intelligence that enables systems to automatically improve performance on tasks through experience and data rather than explicit programming. This definition highlights the core goal of machine learning systems, which is to improve performance through self-learning and adaptation.

The purpose of a machine learning system is to provide a structured, iterative process that encompasses all stages involved in developing, deploying, and maintaining machine learning systems, from problem definition through ongoing monitoring and improvement, as stated in Chapter 17.

Furthermore, machine learning 

In [36]:
# 12. 10 Evaluation Questions
evaluation_questions = [
    "What is retrieval augmented generation?",
    "Why is data preprocessing important in machine learning?",
    "What makes a machine learning system reliable in production?",
    "What is the role of embeddings in semantic search?",
    "How does a transformer process text?",
    "What is the difference between training and inference?",
    "How can a vector database support document search?",
    "What are common causes of model degradation?",
    "How should a machine learning project be evaluated?",
    "What are the benefits and limitations of using large language models?",
]
print(f"Prepared {len(evaluation_questions)} evaluation questions")

Prepared 10 evaluation questions


In [42]:
# 13. Evaluation Table
try:
    import pandas as pd
    evaluation_rows = []
    for question in evaluation_questions:
        result = answer_question(question)
        evaluation_rows.append({
            "question": question,
            "answer": result["answer"],
            "sources": "; ".join(f"{item['metadata']['source']} p.{item['metadata']['page']}" for item in result["sources"]),
            "retrieved_chunks": len(result["sources"]),
        })
    evaluation_table = pd.DataFrame(evaluation_rows)
    display(evaluation_table)
except Exception as error:
    evaluation_table = []
    print(f"Evaluation table requires pandas and a successful pipeline run: {error}")

,question,answer,sources,retrieved_chunks
0,What is retrieval augmented generation?,Retrieval-Augmented Generation (RAG) [Hands-on...,AI Engineering by Chip Huyen.pdf p.501; AI Eng...,4
1,Why is data preprocessing important in machine...,The answer is not supported by the provided do...,Data Preprocessing.pdf p.98; Hands_On_Machine_...,4
2,What makes a machine learning system reliable ...,The provided context does not directly support...,Hands_On_Machine_Learning_with_Scikit_Le.pdf p...,4
3,What is the role of embeddings in semantic sea...,Embeddings are used to turn the search problem...,Hands-on Large Language Models.pdf p.32; Hands...,4
4,How does a transformer process text?,A transformer processes text by taking a text ...,Hands-on Large Language Models.pdf p.96; Hands...,4
5,What is the difference between training and in...,The correct answer is D. Training requires bot...,Machine-Learning-Systems.pdf p.248; Machine-Le...,4
6,How can a vector database support document sea...,According to [AI Engineering by Chip Huyen.pdf...,AI Engineering by Chip Huyen.pdf p.509; design...,4
7,What are common causes of model degradation?,The text does not directly mention the specifi...,AI Engineering by Chip Huyen.pdf p.122; design...,4
8,How should a machine learning project be evalu...,The provided documents do not directly support...,Machine-Learning-Systems.pdf p.2232; Machine-L...,4
9,What are the benefits and limitations of using...,I'm sorry I can't provide a response to that q...,Hands-on Large Language Models.pdf p.47; Hands...,4


In [51]:
# 14. Failure Cases
failure_cases = [
    {"question": "What is the capital of Mars?", "expected": "The corpus should not support an answer."},
    {"question": "Give an answer with no citation.", "expected": "The pipeline should still expose retrieved source metadata."},
    {"question": "Explain a topic absent from these books using certainty.", "expected": "The model should state that the context is insufficient."},
]
for case in failure_cases:
    result = answer_question(case["question"])
    print("Question:", case["question"])
    print("Answer:", result["answer"][:300])
    print("Retrieved:", [item["metadata"] for item in result["sources"]], "\n")

Question: What is the capital of Mars?
Answer: The answer is not supported by the provided documents.
Retrieved: [] 

Question: Give an answer with no citation.
Answer: The answer is not supported by the provided documents.
Retrieved: [] 

Question: Explain a topic absent from these books using certainty.
Answer: The answer is not supported by the provided documents.
Retrieved: [] 



In [52]:
# 15. Persist Vector Store
manifest = {
    "embedding_model": EMBEDDING_MODEL,
    "embedding_backend": embedding_backend,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "chunks": len(chunks),
    "collection": collection_name if collection is not None else None,
    "vector_directory": str(VECTOR_DIR),
}
manifest_path = VECTOR_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Persisted Chroma data to {VECTOR_DIR}")
print(f"Wrote manifest to {manifest_path}")

Persisted Chroma data to E:\Project_RAG\rag-assistant-project\backend\data\vector_store
Wrote manifest to E:\Project_RAG\rag-assistant-project\backend\data\vector_store\manifest.json


In [64]:
test_question = "What is retrieval augmented generation?"
raw_results = retrieve(test_question)
result = answer_question(test_question)

print("Question:", test_question)
print("Raw retrieved chunks:", len(raw_results))
print("Answer:", result["answer"])
print("Sources:", [item["metadata"] for item in result["sources"]])

Question: What is retrieval augmented generation?
Raw retrieved chunks: 4
Answer: Retrieval-Augmented Generation (RAG) is a method for improving the accuracy of large language models (LLMs) in generating factual answers to user queries.

The main idea behind RAG is to use a retrieval algorithm to search for relevant documents or knowledge in a large database and then use the generated text from the LLM to fill in the gaps in the retrieved documents. This approach allows the model to leverage external knowledge and improve its performance in generating accurate answers.

The retrieval algorithms used in RAG are designed to retrieve the most relevant documents based on the query, and then the LLM is used to generate text that can be used to fill in the gaps in the retrieved documents. This approach helps to mitigate the biases and limitations of the LLM in generating accurate answers.

By combining the strengths of both retrieval and generation, RAG aims to improve the accuracy and robus